In [0]:
--Créer la dim_product_category :
--Ecrivez le DDL et chargez la dimension catégorie de produit (dim_product_category) en créant
--autant de niveau de colonnes que nécessaire.
-- Ajouter la clé de la dim_product_category dans la dim_product.

USE CATALOG barbara_lakehouse;
USE SCHEMA gold;

DECLARE OR REPLACE load_date = current_timestamp();
VALUES load_date;

CREATE TABLE IF NOT EXISTS barbara_lakehouse.gold.dim_product_category (
  _tf_dim_product_category_id BIGINT GENERATED ALWAYS AS IDENTITY,

  -- business key (catégorie "leaf", celle du produit)
  prod_category_id            INT,

  -- niveaux (dénormalisés)
  category_level_1_id         INT,
  category_level_1_name       STRING,
  category_level_2_id         INT,
  category_level_2_name       STRING,
  category_level_3_id         INT,
  category_level_3_name       STRING,

  _tf_create_date             TIMESTAMP,
  _tf_update_date             TIMESTAMP
)
USING DELTA;

CREATE OR REPLACE TEMP VIEW _tmp_dim_product_category AS
SELECT
  c1.product_category_id                          AS prod_category_id,
  -- Niveau 1 = racine (grand-parent si existe sinon parent sinon soi)
  COALESCE(c3.product_category_id, c2.product_category_id, c1.product_category_id) AS category_level_1_id,
  COALESCE(c3.name,               c2.name,               c1.name)                  AS category_level_1_name,
  -- Niveau 2 = parent (si existe sinon soi)
  COALESCE(c2.product_category_id, c1.product_category_id) AS category_level_2_id,
  COALESCE(c2.name,                c1.name)                AS category_level_2_name,
  -- Niveau 3 = leaf (soi)
  c1.product_category_id                          AS category_level_3_id,
  c1.name                                         AS category_level_3_name
FROM barbara_lakehouse.silver.productcategory c1
LEFT JOIN barbara_lakehouse.silver.productcategory c2
  ON c1.parent_product_category_id = c2.product_category_id
 AND c2._tf_valid_to IS NULL
LEFT JOIN barbara_lakehouse.silver.productcategory c3
  ON c2.parent_product_category_id = c3.product_category_id
 AND c3._tf_valid_to IS NULL
WHERE c1._tf_valid_to IS NULL;

MERGE INTO barbara_lakehouse.gold.dim_product_category AS tgt
USING _tmp_dim_product_category AS src
ON tgt.prod_category_id = src.prod_category_id
WHEN MATCHED AND (
       tgt.category_level_1_id   IS DISTINCT FROM src.category_level_1_id
    OR tgt.category_level_1_name IS DISTINCT FROM src.category_level_1_name
    OR tgt.category_level_2_id   IS DISTINCT FROM src.category_level_2_id
    OR tgt.category_level_2_name IS DISTINCT FROM src.category_level_2_name
    OR tgt.category_level_3_id   IS DISTINCT FROM src.category_level_3_id
    OR tgt.category_level_3_name IS DISTINCT FROM src.category_level_3_name
) THEN
  UPDATE SET
    tgt.category_level_1_id   = src.category_level_1_id,
    tgt.category_level_1_name = src.category_level_1_name,
    tgt.category_level_2_id   = src.category_level_2_id,
    tgt.category_level_2_name = src.category_level_2_name,
    tgt.category_level_3_id   = src.category_level_3_id,
    tgt.category_level_3_name = src.category_level_3_name,
    tgt._tf_update_date       = load_date
WHEN NOT MATCHED THEN
  INSERT (
    prod_category_id,
    category_level_1_id,
    category_level_1_name,
    category_level_2_id,
    category_level_2_name,
    category_level_3_id,
    category_level_3_name,
    _tf_create_date,
    _tf_update_date
  )
  VALUES (
    src.prod_category_id,
    src.category_level_1_id,
    src.category_level_1_name,
    src.category_level_2_id,
    src.category_level_2_name,
    src.category_level_3_id,
    src.category_level_3_name,
    load_date,
    load_date
  );

ALTER TABLE barbara_lakehouse.gold.dim_product
ADD COLUMN _tf_dim_product_category_id BIGINT;

MERGE INTO barbara_lakehouse.gold.dim_product AS dp
USING (
  SELECT
    dpp._tf_dim_product_id,
    COALESCE(dpc._tf_dim_product_category_id, -9) AS _tf_dim_product_category_id
  FROM barbara_lakehouse.gold.dim_product dpp
  LEFT JOIN barbara_lakehouse.silver.product p
    ON dpp.prod_product_id = p.product_id
   AND p._tf_valid_to IS NULL
  LEFT JOIN barbara_lakehouse.gold.dim_product_category dpc
    ON p.product_category_id = dpc.prod_category_id
  WHERE dpp._tf_valid_to IS NULL
) src
ON dp._tf_dim_product_id = src._tf_dim_product_id
WHEN MATCHED THEN
  UPDATE SET
    dp._tf_dim_product_category_id = src._tf_dim_product_category_id,
    dp._tf_update_date = load_date;

